# PortWatch AI — Ingestion Notebook (Local Version)

**Converted from:** Azure Databricks (Spark + ADLS Gen2)  
**Target:** Local Python + Pandas  

### Changes from original:
- **REMOVED:** All `abfss://` paths → replaced with local `data/` paths
- **REMOVED:** Azure Service Principal config (tenant_id, client_id, client_secret)
- **REMOVED:** `spark.conf.set(...)` ADLS OAuth configuration (Cell 2)
- **REMOVED:** `dbutils.fs.ls(...)` → replaced with `os.listdir()`
- **REMOVED:** `spark.read.csv(...)` → replaced with `pd.read_csv()`
- **REMOVED:** `display(...)` → replaced with `print(df.head())`
- **REMOVED:** All `pyspark.sql.functions` → replaced with Pandas equivalents
- **REMOVED:** `pyspark.sql.window.Window` → replaced with `groupby().shift()`/`.rolling()`
- **REMOVED:** `df.write.parquet(...)` to ADLS → replaced with `df.to_parquet()` locally
- **REMOVED:** `spark.read.parquet(...)` readback → replaced with `pd.read_parquet()`
- **CONSOLIDATED:** Multiple date-parsing attempt cells → single clean `pd.to_datetime()` call
- **CONSOLIDATED:** Redundant `df = df_sample` re-assignments removed

### Output:
- `data/port_daily.parquet` — daily port-call features with lag and rolling averages

In [ ]:
# =============================================================================
# Cell 1 — CONFIG & IMPORTS
# =============================================================================
# REPLACED: Azure storage account config, abfss:// paths, Service Principal secrets
# REMOVED:  storage_account, container_raw, container_meta, container_features
# REMOVED:  tenant_id, client_id, client_secret
# REMOVED:  raw_base  = f"abfss://{container_raw}@{storage_account}.dfs.core.windows.net/"
# REMOVED:  geo_path  = f"abfss://{container_meta}@{storage_account}.dfs.core.windows.net/portwatch/geo/ports.geojson"
# REMOVED:  features_out = f"abfss://{container_features}@{storage_account}.dfs.core.windows.net/port_daily/"
# =============================================================================

import os
import pandas as pd
import numpy as np
from pathlib import Path

# --- Local path config (replaces abfss:// paths) ---
BASE_DIR = Path(".")                              # notebook's working directory
DATA_DIR = BASE_DIR / "data"                      # replaces raw_base (abfss://raw@...)
MODELS_DIR = BASE_DIR / "models"                  # for downstream notebooks
OUTPUTS_DIR = BASE_DIR / "outputs"                # for downstream notebooks

# Input file (was: abfss://raw@pwaidishanth131105.dfs.core.windows.net/Daily_Port_Activity_Data_and_Trade_Estimates.csv)
CSV_PATH = DATA_DIR / "Daily_Port_Activity_Data_and_Trade_Estimates.csv"

# Output file (was: abfss://features@pwaidishanth131105.dfs.core.windows.net/port_daily/)
FEATURES_OUT = DATA_DIR / "port_daily.parquet"

# Ensure directories exist
DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)

print("Paths set:")
print(f"  Input CSV:  {CSV_PATH}")
print(f"  Output:     {FEATURES_OUT}")

In [ ]:
# =============================================================================
# Cell 2 — LIST RAW FILES
# =============================================================================
# REPLACED: dbutils.fs.ls(raw_base) and dbutils.fs.ls(f"abfss://...")
#           → os.listdir() with file size info
# REMOVED:  Entire Spark ADLS OAuth config cell (spark.conf.set for auth.type,
#           oauth.provider.type, oauth2.client.id, oauth2.client.secret,
#           oauth2.client.endpoint)
# =============================================================================

print("Listing files in data directory:", DATA_DIR)
for f in sorted(DATA_DIR.iterdir()):
    size = f.stat().st_size if f.is_file() else "<dir>"
    print(f"  {f.name:60s}  {size}")

# Verify the CSV exists
assert CSV_PATH.exists(), f"ERROR: CSV not found at {CSV_PATH}. Please place the file there."
print(f"\n✓ CSV found: {CSV_PATH} ({CSV_PATH.stat().st_size:,} bytes)")

In [ ]:
# =============================================================================
# Cell 3 — READ CSV
# =============================================================================
# REPLACED: spark.read.option("header", True).option("inferSchema", True)
#           .csv(csv_path).limit(100000)
#           → pd.read_csv() reading the full file
# REMOVED:  .limit(100000) sampling — Pandas reads in-memory, no Spark overhead
# REPLACED: display(df_sample.limit(10)) → print(df.head(10))
# =============================================================================

print("Reading CSV from:", CSV_PATH)
df = pd.read_csv(CSV_PATH)

print(f"Total rows: {len(df):,}")
print(f"Columns ({len(df.columns)}): {list(df.columns)}")
print("\nPreview (first 10 rows):")
print(df.head(10).to_string())

In [ ]:
# =============================================================================
# Cell 4 — PARSE DATES & CAST NUMERIC COLUMNS
# =============================================================================
# CONSOLIDATED: Original had 5 separate cells trying different date formats:
#   - Cell 7:  to_date(to_timestamp(col("date")))               → produced nulls
#   - Cell 9:  to_date(to_timestamp(col("date"), "yyyy/MM/dd HH:mm:ssXXX")) → nulls
#   - Cell 10: to_date(to_timestamp(col("date"), "yyyy/MM/dd HH:mm:ssZ"))   → fallback
#   - Cell 11: regexp_replace to strip timezone, then to_date   → finally worked
#   All consolidated into a single pd.to_datetime() call which handles
#   the format "2019/01/01 00:00:00+00" natively.
#
# REPLACED: pyspark.sql.functions.to_date, to_timestamp, regexp_replace, col
#           → pd.to_datetime()
# REPLACED: col(c).cast("double") loop → pd.to_numeric()
# REPLACED: display(df2.limit(5)) → print(df.head())
# =============================================================================

cols = list(df.columns)
print("Columns detected:", cols)

# --- 1) Normalize date → event_date ---
if 'date' in cols:
    # pd.to_datetime handles "2019/01/01 00:00:00+00" natively (no regex needed)
    df['event_date'] = pd.to_datetime(df['date'], utc=True).dt.date
    df['event_date'] = pd.to_datetime(df['event_date'])  # ensure datetime64 dtype
else:
    raise Exception("No 'date' column found; inspect df.columns and set it manually.")

# --- 2) Detect port ID and port name columns (same logic as original) ---
port_col = 'portid' if 'portid' in cols else next(
    (c for c in cols if 'port' in c.lower() and 'id' in c.lower()), None
)
port_name_col = 'portname' if 'portname' in cols else next(
    (c for c in cols if 'port' in c.lower() and 'name' in c.lower()), None
)
print("Port ID column:", port_col, "Port name column:", port_name_col)

# --- 3) Cast numeric columns (portcalls*, import*, export*) ---
maybe_numeric_prefixes = ['portcalls', 'import', 'export']
numeric_cols = [c for c in cols if any(c.lower().startswith(p) for p in maybe_numeric_prefixes)]
print("Numeric columns to cast:", numeric_cols)

for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# Show preview with event_date, port id, and first few numeric columns
preview_cols = ['event_date', port_col] + numeric_cols[:6]
print("\nPreview (date + port + first 6 numeric cols):")
print(df[preview_cols].head(10).to_string())

In [ ]:
# =============================================================================
# Cell 5 — AGGREGATE & ADD TEMPORAL FEATURES
# =============================================================================
# REPLACED: pyspark.sql.functions.sum → pandas groupby().sum()
# REPLACED: pyspark.sql.window.Window.partitionBy().orderBy()
#           → pandas groupby().shift() and groupby().rolling()
# REPLACED: lag("daily_port_calls", 1).over(w) → groupby().shift(1)
# REPLACED: avg("daily_port_calls").over(w.rowsBetween(-7, -1))
#           → groupby().transform(lambda: shift(1).rolling(7).mean())
# REPLACED: year(col("event_date")), month(col("event_date"))
#           → dt.year, dt.month
# REPLACED: display(port_daily.limit(20)) → print(port_daily.head(20))
# =============================================================================

if port_col is None:
    raise Exception("No port identifier column found. Inspect df.columns and set port_col manually.")

# Pick arrivals column (prefer 'portcalls' aggregate, same logic as original)
if 'portcalls' in cols:
    arrivals_col = 'portcalls'
else:
    arrivals_candidates = [c for c in cols if c.lower().startswith('portcalls_')]
    arrivals_col = arrivals_candidates[0] if arrivals_candidates else None

print("Using port id column:", port_col)
print("Using arrivals column:", arrivals_col)

# --- Compute daily aggregation ---
if arrivals_col:
    port_daily = (
        df.groupby([port_col, 'event_date'], as_index=False)[arrivals_col]
        .sum()
        .rename(columns={arrivals_col: 'daily_port_calls'})
    )
else:
    port_daily = (
        df.groupby([port_col, 'event_date'], as_index=False)
        .size()
        .rename(columns={'size': 'daily_port_calls'})
    )

# Sort by port + date (required for correct lag/rolling calculations)
port_daily = port_daily.sort_values([port_col, 'event_date']).reset_index(drop=True)

# --- Add lag_1: previous day's port calls (per port) ---
port_daily['lag_1'] = port_daily.groupby(port_col)['daily_port_calls'].shift(1)

# --- Add lag_7_avg: rolling 7-day average of port calls (shifted by 1) ---
# Original Spark: avg("daily_port_calls").over(w.rowsBetween(-7, -1))
# rowsBetween(-7, -1) = average of 7 rows before current row (excluding current)
port_daily['lag_7_avg'] = (
    port_daily.groupby(port_col)['daily_port_calls']
    .transform(lambda x: x.shift(1).rolling(window=7, min_periods=1).mean())
)

# --- Add year and month from event_date ---
port_daily['year'] = port_daily['event_date'].dt.year
port_daily['month'] = port_daily['event_date'].dt.month

# Cast daily_port_calls to int (matches original LongType)
port_daily['daily_port_calls'] = port_daily['daily_port_calls'].astype(int)

# Sanity check
print(f"\nport_daily shape: {port_daily.shape}")
print(f"Columns: {list(port_daily.columns)}")
print(f"Dtypes:\n{port_daily.dtypes}")
print("\nSample (first 20 rows):")
print(port_daily.head(20).to_string())

In [ ]:
# =============================================================================
# Cell 6 — WRITE OUTPUT
# =============================================================================
# REPLACED: port_daily.repartition(50).write.mode("overwrite")
#           .partitionBy("year", "month").parquet(features_out)
#           → single df.to_parquet() call
# REMOVED:  Repartitioning (no Spark partitions locally)
# REMOVED:  Partitioned folder output (year=/month=/) — not needed locally
# =============================================================================

print(f"Writing {len(port_daily):,} rows to: {FEATURES_OUT}")
port_daily.to_parquet(FEATURES_OUT, index=False)
print(f"✓ Write complete. File size: {FEATURES_OUT.stat().st_size:,} bytes")

In [ ]:
# =============================================================================
# Cell 7 — VALIDATE READBACK
# =============================================================================
# REPLACED: spark.read.parquet(features_out) → pd.read_parquet()
# REPLACED: display(val.limit(20)) → print(val.head(20))
# =============================================================================

print(f"Reading back from: {FEATURES_OUT}")
val = pd.read_parquet(FEATURES_OUT)

print(f"Readback rows: {len(val):,}")
print(f"Columns: {list(val.columns)}")
print(f"Dtypes:\n{val.dtypes}")
print("\nSample readback (first 20 rows):")
print(val.head(20).to_string())

# Final validation checks
assert len(val) > 0, "ERROR: Readback is empty!"
assert 'event_date' in val.columns, "ERROR: event_date column missing!"
assert val['event_date'].notna().all(), "WARNING: Some event_date values are null!"
print(f"\n✓ Validation passed. {len(val):,} rows with valid event_date.")
print(f"  Date range: {val['event_date'].min()} → {val['event_date'].max()}")
print(f"  Unique ports: {val['portid'].nunique()}")